In [ ]:
import pickle
from pathlib import Path

import getdist as gd
import getdist.plots as gdplots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from emcee.autocorr import integrated_time

from expconfig.synthetic import SynthConfig
from sampling.priors import CompoundPrior, PriorType

In [ ]:
OUTPUTS_PATH = Path().resolve().parent / "outputs"


def get_results_dir(n: int = 0) -> Path:
    """Get the most recent results directory.

    Parameters
    ----------
    n : int, optional
        Index of the results directory to retrieve.
        0 for the most recent, 1 for the second most recent, etc.
        Default is 0.

    Returns
    -------
    Path
        Path to the results directory.
    """
    output_dirs = sorted(
        OUTPUTS_PATH.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True
    )
    return output_dirs[n]

In [ ]:
RESULTS_DIR = get_results_dir(0)

cfg = SynthConfig.load(RESULTS_DIR / "config.yaml")
prior = CompoundPrior.from_dict(cfg.priors.model_dump())
prior_samples = prior.sample(500_000, np.random.default_rng(1234))

In [ ]:
ln_prob = pickle.load(open(RESULTS_DIR / "lnprob_full.pkl", "rb"))
samples = pickle.load(open(RESULTS_DIR / "samples_full.pkl", "rb"))
ln_prob.shape, samples.shape

In [ ]:
acors = np.zeros((ln_prob.shape[0], samples.shape[-1]))

for i, samples_temp in enumerate(samples):
    x = samples_temp
    acors[i, :] = integrated_time(x, quiet=True)

acors

In [ ]:
# grab T=1
ln_prob = ln_prob[0]
samples = samples[0]

ln_prob.shape, samples.shape

In [ ]:
for prior_component in prior.prior_components:
    if prior_component.type == PriorType.WRAPPED_UNIFORM:
        idx = prior_component.indices
        samples[..., idx] = prior_component.prior_fn._wrap(samples[..., idx])

In [ ]:
fig, axes = plt.subplots(prior.n, 1, figsize=(10, 2 * prior.n), sharex=True)
for ax, samples_parameter in zip(axes, samples.transpose(2, 0, 1)):
    ax.plot(samples_parameter, lw=1)
plt.show()

In [ ]:
burn = 20000
thin = min([100, int(np.nanmax(acors[0]) * 2)])
ln_prob = ln_prob[burn::thin]
samples = samples[burn::thin]

with open(RESULTS_DIR / "lnprob.pkl", "wb") as f:
    pickle.dump(ln_prob.reshape(-1), f)
with open(RESULTS_DIR / "samples.pkl", "wb") as f:
    pickle.dump(samples.reshape(-1, prior.n), f)

fig, axes = plt.subplots(1, 1, figsize=(10, 5))
axes.plot(ln_prob, lw=1)
plt.show()

In [ ]:
names = [
    "dA1",
    "dA2-dA1",
    "dC1",
    "dC2-dC1",
    "dF1",
    "dF2-dF1",
    # "eta11",
    # "eta12-eta11",
    # "eta21",
    # "eta22-eta21",
    "r",
]

posterior_samples_gd = gd.MCSamples(
    samples=samples.reshape(-1, prior.n), names=names, labels=names
)
prior_samples_gd = gd.MCSamples(samples=prior_samples, names=names, labels=names)
g = gdplots.getSubplotPlotter()
g.triangle_plot(
    [prior_samples_gd, posterior_samples_gd],
    legend_labels=["Prior", "Posterior"],
    filled=True,
)
g.export(str(RESULTS_DIR / "corner_plot_prior_posterior.png"))

In [ ]:
g.triangle_plot(
    [posterior_samples_gd],
    legend_labels=["Posterior"],
    filled=True,
)
g.export(str(RESULTS_DIR / "corner_plot_posterior.png"))

In [ ]:
from sddr.sddr import fit_marginalised_posterior

model = fit_marginalised_posterior(
    posterior_samples_gd.samples,
    np.arange(prior.n),
    cfg.flow,
    cfg.training,
)

In [ ]:
posterior_samples = model.sample(10_000)
# should check that this looks like the marginals above. If I do stick with looking at posterior predictive distributions then should probably do this straight away and then plot the marginals based on these samples?

posterior_new_samples_gd = gd.MCSamples(
    samples=posterior_samples, names=names, labels=names
)
g = gdplots.getSubplotPlotter()
g.triangle_plot(
    [posterior_new_samples_gd],
    legend_labels=["Posterior"],
    filled=True,
)

In [ ]:
from expconfig.geometry import IC_RADIUS
from expconfig.synthetic import gaussian_noise
from raytracer import BallInShell
from tti.traveltimes import TravelTimeCalculator
from tti.traveltimes.parametrisations import (
    BaseParametriser,
)


def lonlatrad_to_xyz(lonlatrad: np.ndarray) -> np.ndarray:
    """Convert (lon, lat, radius) to Cartesian (x, y, z) coordinates.

    Parameters
    ----------
    lonlatrad : np.ndarray, shape (..., 3)
        Array of longitude (degrees), latitude (degrees), and radius (km).

    Returns
    -------
    xyz : np.ndarray, shape (..., 3)
        Array of Cartesian coordinates in km.
    """
    lon = np.radians(lonlatrad[..., 0])
    lat = np.radians(lonlatrad[..., 1])
    r = lonlatrad[..., 2]

    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)

    return np.stack([x, y, z], axis=-1)


def determine_weights(
    region, ic_in: np.ndarray, path_directions: np.ndarray
) -> np.ndarray:
    """Determine weights for each path based on the distance travelled in each region.

    Parameters
    ----------
    region : CompositeRegion
        The composite region defining the geometry and properties of the inner core.
    ic_in : ndarray, shape (num_paths, 3)
        Entry points of paths into the inner core (longitude (deg), latitude (deg), radius (km)).
    path_directions : ndarray, shape (num_paths, 3)
        Direction vectors for each path.

    Returns
    -------
    weights : ndarray, shape (1, num_segments, num_paths)
        Fractional distance of each path in each segment.  Additional axis for broadcasting with travel time calculator.
    """
    segment_distances = region.ray_distances_per_region(
        lonlatrad_to_xyz(ic_in), path_directions
    )
    total_distances = segment_distances.sum(axis=1)
    weights = segment_distances / total_distances[:, None]
    return weights.T[None, ...]


def forward(ttc: TravelTimeCalculator, params: np.ndarray) -> np.ndarray:
    """Forward model for synthetic IMIC experiment.

    Parameters
    ----------
    ttc : TravelTimeCalculator
        The travel time calculator.
    params : np.ndarray
        Model parameters. Shape (n_samples, n_parameters).
        Parameters are sorted as [A_1, A_2, C_1, C_2, F_1, F_2, eta1_1, eta2_1, eta1_2, eta2_2, r],
        where A, C, F, eta1, eta2 are the usual TTI parameters for each region (IMIC first, then OIC), and r is the radius of IMIC.

    Returns
    -------
    np.ndarray
        Predicted travel times. Shape (n_samples, n_rays).
    """
    tti_params = params[:, :-1]
    imic_radius = params[:, -1]

    invalid_radii = (imic_radius < 0) | (imic_radius > IC_RADIUS)
    if np.any(invalid_radii):
        raise ValueError(
            "Somehow managed to get an invalid model in the posterior samples."
        )

    weights = np.array(
        [
            determine_weights(BallInShell(r, IC_RADIUS), ttc.ic_in, ttc.path_directions)
            for r in imic_radius
        ]
    ).squeeze()  # shape (n_samples, n_cells, npaths)
    ttc.update_weights(weights)
    return ttc(tti_params)


rng = np.random.default_rng(42)

noise_levels: dict[str, float] = {
    "ab": 0.95,
    "bc": 0.63,
    "cd": 0.29,
    "df": 0.95,
}

DATA_FILE = Path().resolve().parent / "data" / "brett2024_ic_traveltimes.parquet"

TO_RADIANS = np.pi / 180.0


class Parametriser(BaseParametriser):
    """Nested parametrisation comparing corresponding Love parameters in 2 layers.

    A1, A2-A1, C1, C2-C1 etc.
    """

    n_model_params_per_segment = 2

    def __init__(self) -> None:
        # self.transformation = np.eye(14, 10)

        # self.transformation[1, 0] = 1.0  # A2
        # self.transformation[3, 2] = 1.0  # C2
        # self.transformation[5, 4] = 1.0  # F2
        # self.transformation[6:10, :] = 0  # L1, L2, N1, N2
        # self.transformation[10, 6] = TO_RADIANS  # eta11
        # self.transformation[11, 6:8] = TO_RADIANS  # eta12
        # self.transformation[12, 8] = TO_RADIANS  # eta21
        # self.transformation[13, 8:10] = TO_RADIANS  # eta22

        self.transformation = np.eye(14, 6)

        self.transformation[1, 0] = 1.0  # A2
        self.transformation[3, 2] = 1.0  # C2
        self.transformation[5, 4] = 1.0  # F2
        self.transformation[6:14, :] = 0  # L1, L2, N1, N2, eta11, eta12, eta21, eta22

    def to_parameters(self, m: np.ndarray):
        """Transform m to love parameters.

        m will be of shape (batch, n_segmentss*n_model_params_per_segment)
        """
        lv = np.matvec(self.transformation, m)
        lv = np.atleast_2d(lv)
        A, C, F, L, N, eta1, eta2 = lv.reshape(7, -1, self.n_model_params_per_segment)
        return A, C, F, L, N, eta1, eta2

    def apply_jacobian(self, grad: np.ndarray) -> np.ndarray:
        raise NotImplementedError("Not using gradients at the moment")


df = pd.read_parquet(DATA_FILE)
ic_in = np.stack(df.in_location.values)
ic_out = np.stack(df.out_location.values)
dt_over_t = (df.delta_t / df.inner_core_travel_time).values

#  The noise levels for each reference phase are given in seconds, so we need to convert them to fractional traveltime perturbations by dividing by the inner core travel time.
# In principle this gives a different sigma for each observation.
sigma = (df["reference_phase"].map(noise_levels) / df["inner_core_travel_time"]).values

ttc = TravelTimeCalculator(
    ic_in=ic_in,
    ic_out=ic_out,
    normalisation=-0.5,
    parametriser=Parametriser(),
)
posterior_predictive = forward(ttc, posterior_samples)
posterior_predictive += gaussian_noise(sigma, rng, posterior_predictive)

In [ ]:
# ensuring that the histogram range is the same for both the posterior predictive and the synthetic data, so that densities are comparable
all_dists = np.vstack([posterior_predictive, dt_over_t[None, :]])
hist_range = (all_dists.min(), all_dists.max())
common_kwargs = {
    "bins": 50,
    "histtype": "step",
    "range": hist_range,
    "density": True,
}
plt.hist(
    posterior_predictive.T,
    **common_kwargs,
    label="Posterior predictive",
    color=np.full(posterior_predictive.shape[0], fill_value="r"),
    alpha=0.5,
)
plt.hist(
    dt_over_t,
    **common_kwargs,
    label="Synthetic data",
    color="k",
)
plt.xlabel("Traveltime perturbation dt/t")
plt.legend()
plt.savefig(RESULTS_DIR / "posterior_predictive.png")

In [ ]:
zeta = np.arccos(ttc.path_directions[:, 2])
zeta[zeta > np.pi / 2] = np.pi - zeta[zeta > np.pi / 2]
zeta = np.degrees(zeta)

In [ ]:
from matplotlib.colors import ListedColormap

N, M = posterior_predictive.shape

# Repeat x for each prediction and flatten predicted values
x_all = np.tile(zeta, N)
p_all = posterior_predictive.ravel()

# 2D histogram: x on horizontal axis, p on vertical axis
xbins = 50
pbins = 50

H, xedges, pedges = np.histogram2d(x_all, p_all, bins=[xbins, pbins], density=False)

# Optional: normalise each x-column so each x has comparable weight
H = H / np.maximum(H.sum(axis=1, keepdims=True), 1)


reds = plt.get_cmap("Reds")
reds_clipped = ListedColormap(reds(np.linspace(0.0, 1, 256)))
reds_clipped.set_under("white")

H[H == 0] = np.nan

plt.figure(figsize=(9, 5))
img = plt.pcolormesh(xedges, pedges, H.T, shading="auto", cmap=reds_clipped)
plt.errorbar(
    zeta,
    dt_over_t,
    yerr=0,
    color="k",
    alpha=0.1,
    label="Data",
    fmt="k.",
)
plt.axhline(c="k")
plt.grid()
plt.xlabel("zeta (degrees)")
plt.ylabel("dt / t")
plt.title("Posterior predictive density")
plt.colorbar(img, label="relative density")
plt.tight_layout()
plt.legend()

plt.savefig(RESULTS_DIR / "posterior_predictive_zeta.png")

In [ ]:
from matplotlib.colors import ListedColormap

N, M = posterior_predictive.shape

# Repeat x for each prediction and flatten predicted values
x_all = np.tile(zeta, N)
p_all = posterior_predictive.ravel()

# 2D histogram: x on horizontal axis, p on vertical axis
xbins = 50
pbins = 50

H, xedges, pedges = np.histogram2d(x_all, p_all, bins=[xbins, pbins], density=False)

# Optional: normalise each x-column so each x has comparable weight
H = H / np.maximum(H.sum(axis=1, keepdims=True), 1)


reds = plt.get_cmap("Reds")
reds_clipped = ListedColormap(reds(np.linspace(0.0, 1, 256)))
reds_clipped.set_under("white")

H[H == 0] = np.nan

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": "polar"})

ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.set_thetamin(0)
ax.set_thetamax(90)
ax.grid(True)
ax.set_ylabel("Delta T / T")
ax.plot(np.linspace(0, np.pi / 2, 100), np.zeros(100), color="black")
ax.text(
    0.85,
    0.85,
    "$\\zeta$ (deg)",
    transform=ax.transAxes,
    ha="center",
    va="center",
    rotation=0,
)
img = ax.pcolormesh(np.radians(xedges), pedges, H.T, shading="auto", cmap=reds_clipped)
ax.errorbar(
    np.radians(zeta),
    dt_over_t,
    yerr=0,
    alpha=0.1,
    label="Data",
    fmt="k.",
)
ax.set_title("Posterior predictive density")
ax.set_ylim(-0.0379376737474989, 0.050655886221718696)
fig.colorbar(
    img,
    ax=ax,
    orientation="horizontal",
    pad=0.1,
    fraction=0.05,
    label="relative density",
)
fig.tight_layout()
fig.legend()

fig.savefig(RESULTS_DIR / "posterior_predictive_zeta_polar.png")

In [ ]:
posterior_samples_gd.getLatex()